# `rustytree` vs `xarray.open_datatree` on NEXRAD KLOT

Side-by-side: open the public NEXRAD KLOT icechunk repo on S3
(`s3://nexrad-arco/KLOT`, anonymous, 107 groups) with two xarray
engines and compare wall time + verify the resulting `DataTree`s are
semantically identical.

**Symmetric input** — both engines accept the same
`icechunk.IcechunkStore` object (`session.store`). rustytree
internally serialises the underlying `Session` via
`PySession.as_bytes()` and rehydrates it inside the Rust core; the
user doesn't have to plumb credentials separately.

- `engine="zarr"` — xarray's stock Zarr/icechunk path. Sequential
  per-group decode, per-time-variable peek for CF datetime dtype
  inference. ~30-50 s cold-cache.
- `engine="rustytree"` — Rust-backed walker. Cross-extension
  Session unwrap (no double-open), cross-group parallelism via
  `try_join_all`, eager fanout for self-named dim coords,
  metadata-only datetime decoding. ~2-3 s cold-cache.

Run this in a freshly-started kernel for a real cold-cache
comparison. Re-running cells in the same kernel will hit warm caches
(both icechunk's session manifest cache and any TLS connection pools)
and the timings will compress.

## Setup

In [1]:
import time

import icechunk
import xarray as xr

URL = "s3://nexrad-arco/KLOT"
OPTS = {"region": "us-east-1", "anon": True}

print(f"xarray {xr.__version__}")
print(f"icechunk {icechunk.__version__}")

xarray 2026.4.0
icechunk 2.0.4


## Open with `engine="zarr"`

xarray's stock path needs an icechunk `Session` first; we hand the
session's `store` to `xr.open_datatree`.

In [2]:
storage = icechunk.s3_storage(
    bucket="nexrad-arco",
    prefix="KLOT",
    region="us-east-1",
    anonymous=True,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")

In [3]:
t = time.perf_counter()
dt_zarr = xr.open_datatree(
    session.store, 
    engine="zarr", 
    consolidated=False,
    zarr_format=3, 
    chunks={}
)
elapsed_zarr = time.perf_counter() - t
print(f"engine='zarr':       {elapsed_zarr:>8,.1f} s   nodes={sum(1 for _ in dt_zarr.subtree)}")

engine='zarr':           62.2 s   nodes=107


In [4]:
ds_zarr = dt_zarr["/VCP-31/sweep_0"].dataset
ds_zarr

<xarray.DatasetView> Size: 933GB
Dimensions:              (vcp_time: 35368, azimuth: 720, range: 1832)
Coordinates:
  * vcp_time             (vcp_time) datetime64[ns] 283kB 2020-01-01T00:09:26....
  * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
    elevation            (azimuth) float64 6kB dask.array<chunksize=(720,), meta=np.ndarray>
    time                 (vcp_time, azimuth) datetime64[ns] 204MB dask.array<chunksize=(1, 720), meta=np.ndarray>
  * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
Data variables:
    ray_elevation_angle  (vcp_time, azimuth) float64 204MB dask.array<chunksize=(1, 720), meta=np.ndarray>
    DBZH                 (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    ZDR                  (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    CCORH                (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    PHIDP                (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    sweep_fixed_angle    (vcp_time) float32 141kB dask.array<chunksize=(1,), meta=np.ndarray>
    RHOHV                (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    sweep_number         (vcp_time) float32 141kB dask.array<chunksize=(1,), meta=np.ndarray>

## Open with `engine="rustytree"`

Same `session.store` input as `engine="zarr"` — symmetric API.
rustytree unwraps the Python `IcechunkStore` to msgpack bytes
(`session._session.as_bytes()`), passes them across the FFI
boundary, and rehydrates the icechunk `Session` inside the Rust
core via `Session::from_bytes`. No second `Repository::open` and
no separate credential plumbing — the user already encoded
everything in their session.

In [5]:
t = time.perf_counter()
dt_rt = xr.open_datatree(
    session.store,
    engine="rustytree",
    chunks={},
)
elapsed_rt = time.perf_counter() - t
print(f"engine='rustytree':  {elapsed_rt:>8,.1f} s   nodes={sum(1 for _ in dt_rt.subtree)}")
print(f"speedup:             {elapsed_zarr/elapsed_rt:>8,.1f}x")

engine='rustytree':       8.0 s   nodes=107
speedup:                  7.7x


## Structural parity

Both engines should produce semantically identical `DataTree`s —
same paths, same `data_vars`, same coords on every node.

In [6]:
rt_paths = sorted(n.path for n in dt_rt.subtree)
zarr_paths = sorted(n.path for n in dt_zarr.subtree)
print(f"paths match: {rt_paths == zarr_paths}  ({len(rt_paths)} nodes)")

mismatches = []
for path in rt_paths:
    rt_ds = dt_rt[path].dataset
    zarr_ds = dt_zarr[path].dataset
    if set(rt_ds.data_vars) != set(zarr_ds.data_vars):
        mismatches.append((path, "data_vars"))
    if set(rt_ds.coords) != set(zarr_ds.coords):
        mismatches.append((path, "coords"))
    if dict(rt_ds.sizes) != dict(zarr_ds.sizes):
        mismatches.append((path, "sizes"))

print(f"structural mismatches: {len(mismatches)}")

paths match: True  (107 nodes)
structural mismatches: 0


## Sample inspection

Pick one VCP and look at its decoded coords + a `.sel()` on the
datetime axis. CF time decoding ran on both engines, so `vcp_time`
is `datetime64[ns]` backed by a pandas `DatetimeIndex` — `.sel()`
with a date string just works.

In [7]:
ds = dt_rt["/VCP-31/sweep_0"].dataset
print("coords:", list(ds.coords))
print("vcp_time dtype:", ds.coords["vcp_time"].dtype)
print("vcp_time index:", type(ds.indexes["vcp_time"]).__name__)
print("range:", ds.coords["vcp_time"].values.min(), "to", ds.coords["vcp_time"].values.max())
ds

coords: ['vcp_time', 'azimuth', 'elevation', 'range', 'time']
vcp_time dtype: datetime64[ns]
vcp_time index: DatetimeIndex
range: 2020-01-01T00:09:26.997000000 to 2022-12-31T23:07:42.491000000


<xarray.DatasetView> Size: 933GB
Dimensions:              (vcp_time: 35368, azimuth: 720, range: 1832)
Coordinates:
  * vcp_time             (vcp_time) datetime64[ns] 283kB 2020-01-01T00:09:26....
  * azimuth              (azimuth) float64 6kB 0.25 0.75 1.25 ... 359.2 359.8
    elevation            (azimuth) float64 6kB dask.array<chunksize=(720,), meta=np.ndarray>
    time                 (vcp_time, azimuth) datetime64[ns] 204MB dask.array<chunksize=(1, 720), meta=np.ndarray>
  * range                (range) float32 7kB 2.125e+03 2.375e+03 ... 4.599e+05
Data variables:
    CCORH                (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    DBZH                 (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    PHIDP                (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    RHOHV                (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    ZDR                  (vcp_time, azimuth, range) float32 187GB dask.array<chunksize=(1, 720, 1832), meta=np.ndarray>
    ray_elevation_angle  (vcp_time, azimuth) float64 204MB dask.array<chunksize=(1, 720), meta=np.ndarray>
    sweep_fixed_angle    (vcp_time) float32 141kB dask.array<chunksize=(1,), meta=np.ndarray>
    sweep_number         (vcp_time) float32 141kB dask.array<chunksize=(1,), meta=np.ndarray>

In [8]:
# Pick a date that's inside the dataset's range and show .sel works
sample_date = str(ds.coords["vcp_time"].values[0])
ds.sel(vcp_time=sample_date, method="nearest").DBZH

<xarray.DataArray 'DBZH' (azimuth: 720, range: 1832)> Size: 5MB
dask.array<getitem, shape=(720, 1832), dtype=float32, chunksize=(720, 1832), chunktype=numpy.ndarray>
Coordinates:
  * azimuth    (azimuth) float64 6kB 0.25 0.75 1.25 1.75 ... 358.8 359.2 359.8
    elevation  (azimuth) float64 6kB dask.array<chunksize=(720,), meta=np.ndarray>
    time       (azimuth) datetime64[ns] 6kB dask.array<chunksize=(720,), meta=np.ndarray>
  * range      (range) float32 7kB 2.125e+03 2.375e+03 ... 4.596e+05 4.599e+05
    vcp_time   datetime64[ns] 8B 2020-01-01T00:09:26.997000
Attributes:
    long_name:      Equivalent reflectivity factor H
    standard_name:  radar_equivalent_reflectivity_factor_h
    units:          dBZ